# Tutorial 08 — LLI/LAM Determination with the Half-Cell Fit

This tutorial demonstrates the use of the LLI/LAM pipeline: from two CheckUp measurements
of the same cell — one fresh (reference), one aged — the following are determined:


**Workflow:**

```
Read raw data  →  Segments + capacity  →  Extract iOCV (charge branch)
      →  Half-cell fit  →  Visual inspection  →  State dicts  →  lli_lam_covered()
```

.

## Step 0 — Imports and Configuration

Three building blocks are needed:

- **`pydpeet` (`eet`)** for reading and preprocessing,
- **`haf_cell_fitting`** for the half-cell fit,
- **`pauses`/`ocv_simple`/`capa_covered`** for rest-voltage anchors and the LLI/LAM calculation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import pydpeet as eet

from pydpeet.process.analyze.extract.haf_cell_fitting import (
    get_best_half_cell_fit,     # half-cell fit including search over the electrode library
    plot_half_cell_match,       # standard plot for visual inspection
    dir_anode, dir_cathode,     # paths to the bundled half-cell library
)
from pydpeet.process.analyze.extract.pauses import extract_pauses
from pydpeet.process.analyze.extract.ocv_simple import pauses_to_ocv_simple
from pydpeet.process.analyze.extract.capa_covered import lli_lam_covered

eet.set_logging_style("ERROR")

# The two measurements: reference (fresh) and aged state of the same cell
PATH_REF  = "../../../input_data/Input_files_checkup/20231129133302-CheckUp-3-7-AM23NMC00020.xlsx"
PATH_AGED = "../../../input_data/Input_files_checkup/20240216101054-CheckUp-3-8-AM23NMC00020.xlsx"

READER_CONFIG  = "neware_8_0_0_516"     # cycler format (see Tutorial 01)
BATTERY_CONFIG = eet.lgm50lt_nmc_4800   # cell configuration (capacity, voltage limits, …)

## Step 1 — Read Raw Data

`eet.read` reads the cycler export into the PyDPEET standard format
(`Test_Time[s]`, `Voltage[V]`, `Current[A]`, …).

In [ ]:
df_raw_ref  = eet.read(config=READER_CONFIG, input_path=PATH_REF)
df_raw_aged = eet.read(config=READER_CONFIG, input_path=PATH_AGED)
print(f"Reference: {len(df_raw_ref):,} samples")
print(f"Aged:      {len(df_raw_aged):,} samples")

## Step 2 — Add Segments and Capacity

The fit later needs a **SOC column**. For this, the measurements are segmented
(`add_primitive_segments`) and capacity along with SOC is added
(`add_capacity`).

In [3]:
df_seq_ref  = eet.add_primitive_segments(df_raw_ref)
df_seq_aged = eet.add_primitive_segments(df_raw_aged)

df_cap_ref  = eet.add_capacity(df_raw_ref, df_seq_ref, neware_bool=True, config=BATTERY_CONFIG)
df_cap_aged = eet.add_capacity(df_raw_aged, df_seq_aged, neware_bool=True, config=BATTERY_CONFIG)

## Step 3 — Extract iOCV

`extract_ocv_iocv` finds the iOCV sequences in the CheckUp. For the fit, the
(`iOCV_type == "Charge"`) branch is used — the fitting model is defined in the
charge direction (anode lithiated, cathode delithiated).

A DataFrame with the columns **`SOC`** and **`Voltage[V]`** is sufficient as fit input.

In [ ]:
def extract_charge_iocv(df_cap):
    # extract iOCV phases and filter to the charge branch
    phases = eet.extract_ocv_iocv(df=df_cap, config=BATTERY_CONFIG, visualize=False)
    df = pd.concat(phases, ignore_index=True)
    df = df.loc[:, ~df.columns.duplicated()]
    return df[df["iOCV_type"] == "Charge"]

df_charge_ref  = extract_charge_iocv(df_cap_ref)
df_charge_aged = extract_charge_iocv(df_cap_aged)

print(f"Reference: {len(df_charge_ref)} iOCV points, SOC {df_charge_ref['SOC'].min():.2f} … {df_charge_ref['SOC'].max():.2f}")
print(f"Aged:      {len(df_charge_aged)} iOCV points, SOC {df_charge_aged['SOC'].min():.2f} … {df_charge_aged['SOC'].max():.2f}")

## Step 4 — Half-Cell Fit

`get_best_half_cell_fit` tries **every anode/cathode combination** of the bundled
half-cell library and returns the best fit (a DataFrame row with the
electrode pair, stoichiometry window `Sol_*`, and fit quality `RMSE[mV]`).

Useful parameters:

| Parameter | Purpose |
|-----------|-------|
| `full_cell_name` | label of the result (appears in the `FullCell` column) |
| `anodes_dir`, `cathodes_dir` | use custom half-cell curves (CSV with columns `Lithiation`, `OCV`) |
| `full_x_col`, `full_y_col` | different column names of the input DataFrame |


In [ ]:
best_fit_ref  = get_best_half_cell_fit(df_charge_ref, full_cell_name="CheckUp_Reference")
best_fit_aged = get_best_half_cell_fit(df_charge_aged, full_cell_name="CheckUp_aged")

fit_overview = pd.concat([best_fit_ref, best_fit_aged], ignore_index=True)
fit_overview.index = ["Reference", "aged"]
fit_overview[["FullCell", "Anode", "Cathode", "RMSE[mV]",
              "Sol_Anode_Min", "Sol_Anode_Max", "Sol_Cathode_Min", "Sol_Cathode_Max"]].round(4)

Both states should fit to **the same electrode pair**.


## Step 5 — Plotting

`plot_half_cell_match` plots the measurement, the simulated full cell, and the two back-calculated
electrode curves (and saves the plot as a PNG in the working directory). What to look out for:

- The **red simulation** should practically cover the **blue measurement**.
- The electrode curves should be physically plausible (anode at the bottom, cathode at the top).

In [ ]:
for label, best_fit, df_charge in [("Reference", best_fit_ref, df_charge_ref),
                                    ("aged", best_fit_aged, df_charge_aged)]:
    plot_half_cell_match(
        full_df        = df_charge,
        anode_df       = pd.read_csv(f"{dir_anode}/{best_fit['Anode'].iloc[0]}"),
        cathode_df     = pd.read_csv(f"{dir_cathode}/{best_fit['Cathode'].iloc[0]}"),
        solution_array = best_fit["Solution_Array"].iloc[0],
        filename       = f"Tutorial08_LLI_LAM_Fit_{label}.png",
    )

Intermediate step: extracting the pauses and OCV points for capacity determination

In [ ]:
def build_state(df_cap, df_charge, best_fit, min_pause_duration_s=60):
    # cut the time window of the iOCV charge sequence from the full measurement
    t0, t1 = df_charge["Test_Time[s]"].min(), df_charge["Test_Time[s]"].max()
    seg = df_cap[(df_cap["Test_Time[s]"] >= t0) & (df_cap["Test_Time[s]"] <= t1)]
    anchors = pauses_to_ocv_simple(extract_pauses(seg, min_pause_duration_s=min_pause_duration_s))
    return {
        "df":         seg,
        "anchors":    anchors,
        "fit_result": best_fit,
        "anode_df":   pd.read_csv(f"{dir_anode}/{best_fit['Anode'].iloc[0]}"),
        "cathode_df": pd.read_csv(f"{dir_cathode}/{best_fit['Cathode'].iloc[0]}"),
    }

state_ref  = build_state(df_cap_ref, df_charge_ref, best_fit_ref)
state_aged = build_state(df_cap_aged, df_charge_aged, best_fit_aged)
print(f"Anchors reference: {len(state_ref['anchors'])}   Anchors aged: {len(state_aged['anchors'])}")

## Step 7 — Calculate LLI/LAM

One call does the rest: `lli_lam_covered(reference, aged)` determines for both states
the capacity over the **jointly covered SOC range** and from that the degradation quantities.

The most important output columns:

| Column | Meaning |
|--------|-----------|
| `Full_Cell_Cap_ref_mAh`, `Full_Cell_Cap_mAh` | measured capacities (reference / aged) |
| `SOH_pct` | capacity aged / reference × 100 |
| `LLI_mAh`, `LLI_pct` | lithium inventory loss (percentage **relative to the inventory**, not to the capacity) |
| `LAM_Anode_pct`, `LAM_Cathode_pct` | active material loss per electrode |
| `SOC_window_lo`, `SOC_window_hi` | joint SOC window used |


In [ ]:
lli_lam_result = lli_lam_covered(state_ref, state_aged)

print(f"Joint SOC window: {lli_lam_result['SOC_window_lo'].iloc[0]:.3f} … "
      f"{lli_lam_result['SOC_window_hi'].iloc[0]:.3f}")
lli_lam_result[["Full_Cell_Cap_ref_mAh", "Full_Cell_Cap_mAh", "SOH_pct",
                "LLI_mAh", "LLI_pct",
                "LAM_Anode_mAh", "LAM_Anode_pct",
                "LAM_Cathode_mAh", "LAM_Cathode_pct"]].round(2).T